# Multiple Integration: Theory, Algorithms, and SymPy Implementation

**Topics covered**
1. Definite integration — review and notation
2. Multiple integrals — definition and applications
3. Multiple vs. iterated integrals (Fubini's theorem)
4. The layer-cake / co-area trick
5. `multiple_integrate` — examples for all nine strategies
6. References


## 0  Setup

In [1]:
import sympy as sp
from sympy import (
    symbols, integrate, oo, pi, exp, sin, cos, sqrt, log,
    Rational, simplify, Abs, Heaviside, Piecewise, sign, diff, gamma, det, Matrix, E
)
sp.init_printing(use_unicode=True)

x, y, z, t, r, theta, phi = symbols('x y z t r theta phi', real=True)
a, b = symbols('a b', positive=True)


## 1  Definite Integration — Review

### 1.1  The Riemann integral

For a bounded function $f:[a,b]\to\mathbb{R}$ the **definite integral** is the limit of Riemann sums:

$$\int_a^b f(x)\,dx \;=\; \lim_{n\to\infty}\sum_{k=1}^n f(x_k^*)\,\Delta x_k$$

### 1.2  The Fundamental Theorem of Calculus

If $F'=f$ on $[a,b]$:

$$\int_a^b f(x)\,dx = F(b)-F(a)$$

### 1.3  Improper integrals

When the domain or integrand is unbounded the integral is a limit:

$$\int_a^\infty f(x)\,dx = \lim_{R\to\infty}\int_a^R f(x)\,dx$$

**Classic example** — the Gaussian integral, proved by squaring and converting to polar coordinates:

$$\int_{-\infty}^\infty e^{-x^2}\,dx = \sqrt{\pi}$$


### 1.1  What a definite integral means

For a continuous function $f$ on an interval $[a,b]$, the definite integral

$$
\int_a^b f(x)\,dx
$$

measures the signed accumulation of $f$ over the interval.  Depending on context, the same quantity may be interpreted as:

- signed area under a curve,
- accumulated mass when $f$ is a density,
- total change when $f$ is a rate,
- probability mass when $f$ is a probability density.

The fundamental theorem of calculus gives the bridge between antiderivatives and definite integrals:

$$
\int_a^b f(x)\,dx = F(b)-F(a), \qquad F'(x)=f(x).
$$

In symbolic integration, this is only part of the story: one must also decide whether the antiderivative exists in a usable closed form, whether the integral converges, and whether a change of variables or a special-function representation is more natural than a direct antiderivative.

### 1.2  Standard tools for definite integrals

A few recurring principles show up throughout this notebook.

**Linearity**
$$
\int_a^b (\alpha f(x)+\beta g(x))\,dx
=
\alpha \int_a^b f(x)\,dx + \beta \int_a^b g(x)\,dx.
$$

**Additivity over intervals**
$$
\int_a^b f(x)\,dx = \int_a^c f(x)\,dx + \int_c^b f(x)\,dx.
$$

**Symmetry on symmetric domains**
If $f$ is odd, then
$$
\int_{-a}^{a} f(x)\,dx = 0.
$$
If $f$ is even, then
$$
\int_{-a}^{a} f(x)\,dx = 2\int_0^a f(x)\,dx.
$$

**Substitution**
If $x=\phi(u)$ is smooth and monotone, then
$$
\int_a^b f(x)\,dx
=
\int_{\phi^{-1}(a)}^{\phi^{-1}(b)} f(\phi(u))\,\phi'(u)\,du.
$$

**Integration by parts**
$$
\int_a^b u(x)\,v'(x)\,dx
=
[u(x)v(x)]_a^b - \int_a^b u'(x)\,v(x)\,dx.
$$

These rules remain important in the multivariable setting, but the domain geometry becomes an additional central ingredient.

### 1.3  Proper vs. improper integrals

Not every definite integral is taken over a finite interval or with a bounded integrand.  Typical improper integrals include

$$
\int_1^\infty \frac{dx}{x^2},
\qquad
\int_0^1 \frac{dx}{\sqrt{x}},
\qquad
\int_{-\infty}^{\infty} e^{-x^2}\,dx.
$$

These are defined by limits.  For example,

$$
\int_1^\infty \frac{dx}{x^2}
=
\lim_{R\to\infty}\int_1^R \frac{dx}{x^2}.
$$

Convergence is part of the mathematical problem, not something automatic.  A symbolic integration engine therefore has to distinguish at least three situations:

1. the integral converges and has an exact closed form,
2. the integral diverges,
3. the integral may require a special interpretation (principal value, oscillatory regularization, and so on).

This notebook focuses on ordinary convergence and exact symbolic evaluation.

In [2]:
# Verify the Gaussian integral
integrate(exp(-x**2), (x, -oo, oo))


In [3]:
# A selection of closed-form definite integrals
for expr, label in [
    (x**3,           "[0,2] polynomial"),
    (sin(x),         "[0,π] trig"),
    (exp(-a*x),      "[0,∞) exponential"),
    (1/(1 + x**2),   "(-∞,∞) rational"),
    (log(x),         "[0,1] log singularity"),
]:
    result = integrate(expr, (x, 0, 2) if label.startswith("[0,2]") else
                             (x, 0, pi) if "π]" in label else
                             (x, 0, oo) if "∞)" in label else
                             (x, -oo, oo) if "(-∞" in label else
                             (x, 0, 1))
    print(f"{label:25s}: {result}")


[0,2] polynomial         : 4
[0,π] trig               : 2
[0,∞) exponential        : 1/a
(-∞,∞) rational          : pi/2
[0,1] log singularity    : -1


## 2  Multiple Integrals

### 2.1  Definition

Let $\Omega\subseteq\mathbb{R}^n$ be measurable and $f:\Omega\to\mathbb{R}$ integrable. Partitioning $\Omega$ into sub-regions $\{\Omega_k\}$ with volumes $|\Omega_k|$:

$$\int_\Omega f(\mathbf{x})\,d\mathbf{x} \;=\; \lim_{\|\text{partition}\|\to 0}\sum_k f(\mathbf{x}_k^*)\,|\Omega_k|$$

### 2.2  Applications

| Application | Integral |
|---|---|
| **Area / Volume** | $\int_\Omega 1\,d\mathbf{x}$ |
| **Mass** | $\int_\Omega \rho(\mathbf{x})\,d\mathbf{x}$ |
| **Centre of mass** | $\bar{x}_i = \frac{1}{M}\int_\Omega x_i\,\rho\,d\mathbf{x}$ |
| **Moment of inertia** | $I=\int_\Omega r^2\rho\,d\mathbf{x}$ |
| **Probability** | $\Pr[X\in A]=\int_A f_X\,d\mathbf{x}$ |
| **Partition function** | $Z=\int e^{-\beta H}\,d\mathbf{x}$ |

### 2.3  Coordinate changes

For a diffeomorphism $\mathbf{x}=\Phi(\mathbf{u})$:

$$\int_\Omega f(\mathbf{x})\,d\mathbf{x} = \int_{\Phi^{-1}(\Omega)} f(\Phi(\mathbf{u}))\,|\det J_\Phi|\,d\mathbf{u}$$

**Polar** $(r,\theta)$: $|J|=r$ &emsp; **Cylindrical** $(r,\theta,z)$: $|J|=r$ &emsp; **Spherical** $(r,\theta,\phi)$: $|J|=r^2\sin\theta$


### 2.1  Multiple integrals as integrals over regions

A multiple integral is an integral over a subset $R \subset \mathbb{R}^n$:

$$
\int_R f(\mathbf{x})\,d\mathbf{x}.
$$

For $n=2$, one usually writes
$$
\iint_R f(x,y)\,dA,
$$
and for $n=3$,
$$
\iiint_R f(x,y,z)\,dV.
$$

Geometrically, $f$ may represent:

- a density,
- a temperature field,
- a probability density,
- a charge or mass distribution,
- or simply an integrand whose total accumulation over the region is sought.

The region $R$ is just as important as the integrand.  A rectangle, a triangle, a disk, and all of $\mathbb{R}^n$ lead to very different exact methods even when the integrand itself looks similar.

### 2.2  Product regions and iterated integrals

On a rectangular region
$$
R = [a_1,b_1]\times\cdots\times[a_n,b_n],
$$
one often rewrites the multiple integral as an iterated integral:

$$
\int_R f(\mathbf{x})\,d\mathbf{x}
=
\int_{a_1}^{b_1}\cdots\int_{a_n}^{b_n} f(x_1,\dots,x_n)\,dx_n\cdots dx_1.
$$

When absolute integrability holds, Fubini's theorem says that the order of integration may be changed without affecting the result.  Tonelli's theorem gives a related statement for nonnegative integrands.

This matters computationally because different orders of integration can have very different symbolic complexity.  One order may expose a Gaussian, beta, or rational integral immediately, while another may produce difficult intermediate expressions.

### 2.3  Non-rectangular regions and dependent bounds

Not all multiple integrals live on product regions.  A triangle such as

$$
R=\{(x,y): 0\le x\le 1,\; 0\le y\le 1-x\}
$$

is still perfectly standard, and may be written as

$$
\int_0^1\int_0^{1-x} f(x,y)\,dy\,dx.
$$

The inner bound depends on the outer variable, so the geometry of the region is encoded directly into the iterated integral.  In general, changing the order of integration then requires rewriting the region, and that can be easy, moderate, or difficult depending on the boundary equations.

This notebook uses such examples to show the distinction between:
- a multiple integral as a geometric object,
- an iterated integral as one chosen description of that object.

### 2.4  Change of variables and Jacobians

For many exact multiple integrals, the key step is a coordinate transformation.  If $\mathbf{x}=\Phi(\mathbf{u})$ is a smooth bijection with Jacobian matrix $D\Phi$, then

$$
\int_R f(\mathbf{x})\,d\mathbf{x}
=
\int_{\Phi^{-1}(R)} f(\Phi(\mathbf{u}))\,\left|\det D\Phi(\mathbf{u})\right|\,d\mathbf{u}.
$$

The factor
$$
\left|\det D\Phi(\mathbf{u})\right|
$$
is the Jacobian determinant.

Examples:
- polar coordinates contribute a factor of $r$,
- cylindrical coordinates contribute a factor of $r$,
- spherical coordinates contribute a factor of $r^2\sin\theta$.

In symbolic computation, a good change of variables can turn a difficult region into a simple one, or can transform a coupled integrand into a separable product.

### 2.5  Families that often admit exact evaluation

A large fraction of exact multiple-integration problems fall into a handful of recognizable families:

- **box moments** on rectangular domains,
- **simplex moments** on triangular or simplex-shaped regions,
- **Gaussian moments** on $\mathbb{R}^n$,
- **radial integrals** over disks and balls,
- **separable product integrals**,
- **beta/gamma-type integrals**,
- **rational integrals** with contour or residue methods,
- **layer-cake or co-area reductions** for $f(g(\mathbf{x}))$.

The `multiple_integrate` function tries to recognize several of these families before falling back to more generic symbolic iteration.

In [4]:
# Volume of a ball of radius R in spherical coordinates
R = symbols('R', positive=True)
vol_ball = integrate(r**2 * sin(theta), (r, 0, R), (theta, 0, pi), (phi, 0, 2*pi))
print("Volume of ball:", vol_ball)   # 4πR³/3


Volume of ball: 4*pi*R**3/3


In [5]:
# Mass of unit disk with density ρ(r) = 1 - r²
mass = integrate((1 - r**2) * r, (r, 0, 1), (theta, 0, 2*pi))
print("Mass of unit disk:", mass)    # π/2


Mass of unit disk: pi/2


## 3  Multiple Integrals vs. Iterated Integrals

### 3.1  The distinction

A **multiple integral** $\int_\Omega f\,d\mathbf{x}$ is defined directly as a Riemann-sum limit.

An **iterated integral** reduces it to a sequence of 1-D integrals:

$$\int_a^b\!\left[\int_{g(x)}^{h(x)} f(x,y)\,dy\right]dx$$

### 3.2  Fubini's theorem

> **Fubini (1907).** If $f\in L^1([a,b]\times[c,d])$:
> $$\iint f\,dA = \int_a^b\!\int_c^d f\,dy\,dx = \int_c^d\!\int_a^b f\,dx\,dy$$

Fubini is the **licence** to compute multiple integrals as iterated ones.

### 3.3  When Fubini fails

Fubini requires $\int|f|<\infty$. The classic counterexample:

$$f(x,y)=\frac{x^2-y^2}{(x^2+y^2)^2}$$

gives $\int_0^1\!\int_0^1 f\,dy\,dx=\pi/4$ but $\int_0^1\!\int_0^1 f\,dx\,dy=-\pi/4$.

### 3.4  Order of integration on non-rectangular domains

On a triangle $\{0\le x\le 1,\;0\le y\le 1-x\}$:

$$\iint f\,dA = \int_0^1\!\int_0^{1-x}f\,dy\,dx = \int_0^1\!\int_0^{1-y}f\,dx\,dy$$


In [6]:
# Fubini check — both orders must agree
f_xy = x**2 * y + x * y**3
order1 = integrate(integrate(f_xy, (y, 0, 1)), (x, 0, 1))
order2 = integrate(integrate(f_xy, (x, 0, 1)), (y, 0, 1))
print("dy dx:", order1, "  dx dy:", order2, "  equal:", simplify(order1 - order2) == 0)


dy dx: 7/24   dx dy: 7/24   equal: True


In [7]:
# Switching integration order on a triangle
order_A = integrate(integrate(x + y, (y, 0, x)),   (x, 0, 1))
order_B = integrate(integrate(x + y, (x, y, 1)),   (y, 0, 1))
print("Order A:", order_A, "  Order B:", order_B, "  equal:", simplify(order_A - order_B) == 0)


Order A: 1/2   Order B: 1/2   equal: True


## 4  The Layer-Cake / Co-Area Trick

`multiple_integrate` reduces an $n$-dimensional integral to a **one-dimensional one**
for integrands of the form $f(g(\mathbf{x}))$.  
**Crucially, $g$ can be any measurable function — polynomial, trigonometric,
exponential, or discontinuous.**

### 4.1  The master identity

Define the **cumulative measure** of $g$ over $\Omega$:

$$\mu(y) = \lambda\!\left(\{\mathbf{x}\in\Omega : g(\mathbf{x})\le y\}\right)
= \int_\Omega \Theta(y - g(\mathbf{x}))\,d\mathbf{x}$$

where $\Theta$ is the Heaviside step function. Then:

$$\boxed{
\int_\Omega f(g(\mathbf{x}))\,d\mathbf{x} = \int_{y_{\min}}^{y_{\max}} f(y)\,\mu'(y)\,dy
}$$

This follows from Fubini's theorem and requires **nothing special about $g$**.

### 4.2  The co-area formula (Federer, 1959)

$$\mu'(y) = \int_{g^{-1}(y)} \frac{1}{|\nabla g(\mathbf{x})|}\,d\mathcal{H}^{n-1}(\mathbf{x})$$

For a **single-variable monotone** $g$, the level set $g^{-1}(y)$ is one point $x(y)$, so:

$$\mu'(y) = \frac{1}{|g'(x(y))|} = \left|\frac{dx}{dy}\right|$$

For **piecewise-monotone** $g$ with branches $\{x_k\}$ at each level $y$:

$$\mu'(y) = \sum_k \frac{1}{|g'(x_k)|}$$

### 4.3  Special cases

| $g$ | Domain | $\mu'(y)$ | Strategy |
|---|---|---|---|
| $\mathbf{b}\cdot\mathbf{x}+c$ (linear) | $[0,\infty)^n$ | $\frac{(y-c)^{n-1}}{\prod b_i\,(n-1)!}$ | S1 |
| $\mathbf{x}^\top A\mathbf{x}$ (quadratic) | $\mathbb{R}^n$ | $\frac{\pi^{n/2}}{\sqrt{\det A}\,\Gamma(n/2+1)}\cdot\frac{n}{2}(y-y_{\min})^{n/2-1}$ | S2/S3 |
| any polynomial | bounded $\Omega$ | computed by SymPy from $\int\Theta(y-g)$ | S4 |
| $h_1(x_1)+\cdots+h_n(x_n)$ | any | convolution $\nu_1*\cdots*\nu_n$ | S5 |
| monotone $g(x_i)$ | $[a_i,b_i]$ | $\lvert dx/dy\rvert$ via $g^{-1}$ | S6 |
| piecewise-monotone $g(x_i)$ | $[a_i,b_i]$ | $\sum_k 1/\lvert g'(x_k)\rvert$ | S7 |
| non-polynomial multivariate | bounded | SymPy evaluates $\int\Theta(y-g)$ | S8 |

### 4.4  Step-by-step walkthrough — polynomial $g$

We demonstrate the general Heaviside approach (Strategy 4) on $\iint_{[0,1]^2}(x+y)^3\,dA$.


In [8]:
# The level sets of g = x+y on [0,1]^2 are line segments.
# We derive mu(y) analytically:
#   For y in [0,1]: {x+y <= y} is the triangle below the line, area = y^2/2
#   For y in [1,2]: area = 1 - (2-y)^2/2
ys = symbols('ys', real=True)

mu_poly = Piecewise(
    (ys**2 / 2,           (ys >= 0) & (ys <= 1)),
    (1 - (2 - ys)**2 / 2, (ys > 1)  & (ys <= 2)),
    (0, True)
)
print("μ(y) =", mu_poly)


μ(y) = Piecewise((ys**2/2, (ys >= 0) & (ys <= 1)), (1 - (2 - ys)**2/2, (ys <= 2) & (ys > 1)), (0, True))


In [9]:
# Step 2: differentiate to get the pushforward density μ'(y)
density_poly = simplify(diff(mu_poly, ys))
print("μ'(y) =", density_poly)
# Triangular density: rises linearly to 1 at y=1, falls back to 0 at y=2


μ'(y) = Piecewise((ys, (ys >= 0) & (ys <= 1)), (2 - ys, (ys <= 2) & (ys > 1)), (0, True))


In [10]:
# Step 3: ∫ f(y)·μ'(y) dy  with f(t) = t³
result_layercake = integrate(ys**3 * density_poly, (ys, 0, 2))
result_direct    = integrate(integrate((x + y)**3, (y, 0, 1)), (x, 0, 1))
print("Layer-cake:", result_layercake)   # 3/2
print("Direct:    ", result_direct)      # 3/2
print("Match:", simplify(result_layercake - result_direct) == 0)


Layer-cake: 3/2
Direct:     3/2
Match: True


### 4.5  Step-by-step walkthrough — non-polynomial $g$

The same identity works for $g = \sin(x)$ on $[0,\pi]$, a piecewise-monotone function.

We derive $\mu(y)$ analytically. For $y \in [0,1]$, the set
$\{x\in[0,\pi] : \sin(x) \le y\}$ consists of two pieces:
$[0, \arcsin y]$ and $[\pi - \arcsin y, \pi]$, with total length $2\arcsin y$.
Differentiating:

$$\mu'(y) = \frac{2}{\sqrt{1-y^2}}, \qquad y \in [0,1]$$

This is the sum of two co-area branches: $1/|\sin'(x_1)| + 1/|\sin'(x_2)|
= 1/\cos(\arcsin y) + 1/\cos(\pi-\arcsin y) = 2/\sqrt{1-y^2}$.


In [11]:
# μ(y) for g = sin(x) on [0, π] derived analytically
ys2 = symbols('ys2', real=True)

mu_sin = Piecewise(
    (0,           ys2 < 0),
    (2*sp.asin(ys2), (ys2 >= 0) & (ys2 <= 1)),
    (pi,          ys2 > 1)
)
density_sin = simplify(diff(mu_sin, ys2))
print("μ'(y) =", density_sin)   # 2/sqrt(1 - y^2) on [0,1]

# For f = identity: ∫_0^1 y · μ'(y) dy should equal ∫_0^π sin(x) dx = 2
result_sin = integrate(ys2 * density_sin, (ys2, 0, 1))
print("Layer-cake ∫sin(x):", result_sin)   # 2
print("Direct     ∫sin(x):", integrate(sin(x), (x, 0, pi)))  # 2
print("Match:", simplify(result_sin - 2) == 0)


μ'(y) = Piecewise((0, ys2 < 0), (2/sqrt(1 - ys2**2), ys2 <= 1), (0, True))
Layer-cake ∫sin(x): 2
Direct     ∫sin(x): 2
Match: True


## 5  Examples with `multiple_integrate`

The library implements all nine strategies. Place `multiple_integrate` on your
`PYTHONPATH` (or install with `pip install -e .` from the project root):


In [12]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Using src path:", SRC)
print("Package expected at:", SRC / "multiple_integrate")
from multiple_integrate import multiple_integrate


Project root: /Users/bbhatt/git/Portfolio/SymbolicMath/MultipleIntegrate
Using src path: /Users/bbhatt/git/Portfolio/SymbolicMath/MultipleIntegrate/src
Package expected at: /Users/bbhatt/git/Portfolio/SymbolicMath/MultipleIntegrate/src/multiple_integrate


Before diving into the strategy-specific examples, it is helpful to keep the overall philosophy in mind.  The solver is not a single monolithic rule.  Instead, it tries to identify structure such as:

- linear or quadratic forms,
- Gaussian-type weights,
- product or separable structure,
- monotone substitutions,
- radial or moment families,
- and, as a last resort, plain iterated integration.

The examples below are grouped by the dominant exact-evaluation idea rather than by increasing difficulty.  In practice, many integrals fit more than one pattern.

### 5.1  Strategy 1 — Linear polynomial over $[0,\infty)^n$

**Formula:** $\displaystyle\int_{[0,\infty)^n} f(\mathbf{b}\cdot\mathbf{x}+c)\,d\mathbf{x}
= \frac{1}{\prod b_i\,(n-1)!}\int_c^\infty (y-c)^{n-1}f(y)\,dy$

Fires when $g$ is a *purely linear* polynomial and every range is $[0,\infty)$.


In [13]:
x, y, z = symbols('x y z', real=True)

# n=1: ∫_0^∞ exp(-(2x+1)) dx = exp(-1)/2
s1a = multiple_integrate(exp(-(2*x + 1)), (x, 0, oo))
print("∫₀^∞ exp(-(2x+1)) dx =", s1a)   # exp(-1)/2

# n=2: ∫∫_[0,∞)² exp(-(3x+2y)) dx dy = 1/6
s1b = multiple_integrate(exp(-(3*x + 2*y)), (x, 0, oo), (y, 0, oo))
print("∫∫ exp(-(3x+2y))    =", s1b)   # 1/6

# n=3: ∫∫∫_[0,∞)³ exp(-(x+y+z)) dV = 1
s1c = multiple_integrate(exp(-(x+y+z)), (x, 0, oo), (y, 0, oo), (z, 0, oo))
print("∫∫∫ exp(-(x+y+z))  =", s1c)   # 1


∫₀^∞ exp(-(2x+1)) dx = exp(-1)/2
∫∫ exp(-(3x+2y))    = 1/6
∫∫∫ exp(-(x+y+z))  = 1


### 5.2  Strategy 2 — Quadratic, doubly-infinite

**Formula:** $\displaystyle\int_{\mathbb{R}^n} f(\mathbf{x}^\top A\mathbf{x}+\mathbf{b}\cdot\mathbf{x}+c)\,d\mathbf{x}
= \frac{\pi^{n/2}}{\sqrt{\det A}\,\Gamma(n/2+1)}\int_{y_{\min}}^\infty \frac{n}{2}(y-y_{\min})^{n/2-1}f(y)\,dy$

Fires when $g$ is quadratic with positive-definite $A$ and all ranges are $(-\infty,\infty)$.


In [14]:
# 2-D isotropic Gaussian
s2a = multiple_integrate(exp(-(x**2 + y**2)), (x, -oo, oo), (y, -oo, oo))
print("∫∫_ℝ² exp(-(x²+y²))         =", s2a)   # π

# Anisotropic: ∫∫ exp(-(2x²+3y²)) = π/√6
s2b = multiple_integrate(exp(-(2*x**2 + 3*y**2)), (x, -oo, oo), (y, -oo, oo))
print("∫∫_ℝ² exp(-(2x²+3y²))       =", s2b)   # π/√6

# 3-D Gaussian
s2c = multiple_integrate(exp(-(x**2+y**2+z**2)), (x,-oo,oo),(y,-oo,oo),(z,-oo,oo))
print("∫∫∫_ℝ³ exp(-(x²+y²+z²))     =", s2c)   # π^(3/2)

# With shift: value is still π
s2d = multiple_integrate(exp(-((x-1)**2 + (y+2)**2)), (x,-oo,oo),(y,-oo,oo))
print("∫∫_ℝ² exp(-((x-1)²+(y+2)²)) =", s2d)   # π


∫∫_ℝ² exp(-(x²+y²))         = pi
∫∫_ℝ² exp(-(2x²+3y²))       = sqrt(6)*pi/6
∫∫∫_ℝ³ exp(-(x²+y²+z²))     = pi**(3/2)
∫∫_ℝ² exp(-((x-1)²+(y+2)²)) = pi*erfc(2)/2 + pi*(2 - erfc(2))/2


### 5.3  Strategy 3 — Quadratic, even, half-infinite

Same quadratic engine as S2, but some dimensions have range $[0,\infty)$.
Requires $f\circ g$ to be even in those dimensions; result is halved for each.


In [15]:
# Half-line Gaussian: ∫_0^∞ exp(-x²) dx = √π/2
s3a = multiple_integrate(exp(-x**2), (x, 0, oo))
print("∫₀^∞ exp(-x²)          =", s3a)   # √π/2

# Quarter-plane: ∫∫_[0,∞)² exp(-(x²+y²)) = π/4
s3b = multiple_integrate(exp(-(x**2+y**2)), (x, 0, oo), (y, 0, oo))
print("∫∫_[0,∞)² exp(-(x²+y²)) =", s3b)   # π/4

# Mixed full/half: π/2
s3c = multiple_integrate(exp(-(x**2+y**2)), (x, -oo, oo), (y, 0, oo))
print("∫_ℝ∫₀^∞ exp(-(x²+y²))  =", s3c)   # π/2


∫₀^∞ exp(-x²)          = sqrt(pi)/2
∫∫_[0,∞)² exp(-(x²+y²)) = pi/4
∫_ℝ∫₀^∞ exp(-(x²+y²))  = pi/2


### 5.4  Strategy 4 — General polynomial, Heaviside layer-cake

For polynomial $g$ on any bounded or semi-infinite domain.
SymPy integrates $\Theta(y-g(\mathbf{x}))$ dimension by dimension to obtain $\mu(y)$.


In [16]:
# Bounded domain
s4a = multiple_integrate(x**2 * y, (x, 0, 1), (y, 0, 2))
print("∫∫ x²y  [0,1]×[0,2]      =", s4a)   # 2/3

# (x+y)³ on unit square
s4b = multiple_integrate((x+y)**3, (x, 0, 1), (y, 0, 1))
print("∫∫ (x+y)³  [0,1]²        =", s4b)

# Triangle domain with variable upper limit
s4c = multiple_integrate(x**2 * y, (y, 0, 1-x), (x, 0, 1))
print("∫∫ x²y  triangle         =", s4c)   # 1/60

# Triple integral
s4d = multiple_integrate(x*y*z, (x,0,1),(y,0,1),(z,0,1))
print("∫∫∫ xyz  [0,1]³          =", s4d)   # 1/8


∫∫ x²y  [0,1]×[0,2]      = 2/3
∫∫ (x+y)³  [0,1]²        = 3/2
∫∫ x²y  triangle         = 1/60
∫∫∫ xyz  [0,1]³          = 1/8


### 5.5  Strategy 5 — Separable non-polynomial $g$

**When $g(x_1,\ldots,x_n) = h_1(x_1)+\cdots+h_n(x_n)$** (each term depends on one variable),
the pushforward density of the sum is the **convolution** of the marginal densities:

$$\mu'(y) = (\nu_1 * \nu_2 * \cdots * \nu_n)(y)$$

Each $\nu_i$ is computed via a 1-D Heaviside integral. This works for **any** single-variable
functions $h_i$ — trig, exponential, logarithm, etc.


In [17]:
# Trig sum: ∫∫_[0,π]² cos(x+y) dx dy = -4
s5a = multiple_integrate(cos(x + y), (x, 0, pi), (y, 0, pi))
print("∫∫ cos(x+y)  [0,π]²      =", s5a)   # -4

# Exponential sum: ∫∫_[0,∞)² exp(-(x+y)) = 1
s5b = multiple_integrate(exp(-(x + y)), (x, 0, oo), (y, 0, oo))
print("∫∫ exp(-(x+y))  [0,∞)²  =", s5b)   # 1

# Squared trig sum
s5c = multiple_integrate((sin(x) + sin(y))**2, (x, 0, 1), (y, 0, 1))
print("∫∫ (sin x + sin y)²     =", simplify(s5c))

# 3-D trig sum
s5d = multiple_integrate(sin(x + y + z), (x, 0, 1), (y, 0, 1), (z, 0, 1))
print("∫∫∫ sin(x+y+z)  [0,1]³  =", simplify(s5d))


∫∫ cos(x+y)  [0,π]²      = -4
∫∫ exp(-(x+y))  [0,∞)²  = 1
∫∫ (sin x + sin y)²     = -4*cos(1) - sin(2)/2 + cos(2) + 4
∫∫∫ sin(x+y+z)  [0,1]³  = -1 + cos(3) - 3*cos(2) + 3*cos(1)


### 5.6  Strategy 6 — Monotone substitution

For **single-variable non-polynomial $g$** with no interior critical points,
the co-area density is $\mu'(y) = |dx/dy| = 1/|g'(g^{-1}(y))|$.

The library inverts $g$ analytically via `sympy.solve` and integrates $f(y)|dx/dy|$.
This handles exponentials, logarithms, rational functions, algebraic functions, and more.


In [18]:
# Exponential: ∫_0^1 exp(x) dx = e - 1
s6a = multiple_integrate(exp(x), (x, 0, 1))
print("∫₀¹ exp(x)       =", s6a)   # E - 1

# Logarithm: ∫_1^e log(x) dx = 1
s6b = multiple_integrate(log(x), (x, 1, E))
print("∫₁ᵉ log(x)       =", s6b)   # 1

# Integrable log singularity: ∫_0^1 log(x) dx = -1
s6c = multiple_integrate(log(x), (x, 0, 1))
print("∫₀¹ log(x)       =", s6c)   # -1

# Arctan: ∫_0^1 1/(1+x²) dx = π/4
s6d = multiple_integrate(1/(1 + x**2), (x, 0, 1))
print("∫₀¹ 1/(1+x²)     =", s6d)   # π/4

# Algebraic: ∫_0^4 √x dx = 16/3
s6e = multiple_integrate(sqrt(x), (x, 0, 4))
print("∫₀⁴ √x           =", s6e)   # 16/3

# Half-line (monotone decreasing): ∫_0^∞ exp(-x) dx = 1
s6f = multiple_integrate(exp(-x), (x, 0, oo))
print("∫₀^∞ exp(-x)     =", s6f)   # 1

# With free dimension: ∫₀¹∫₀¹ exp(-x) dx dy = 1 - 1/e
s6g = multiple_integrate(exp(-x), (x, 0, 1), (y, 0, 1))
print("∫₀¹∫₀¹ exp(-x)   =", s6g)   # 1 - 1/e


∫₀¹ exp(x)       = -1 + E
∫₁ᵉ log(x)       = 1
∫₀¹ log(x)       = -1
∫₀¹ 1/(1+x²)     = pi/4
∫₀⁴ √x           = 16/3
∫₀^∞ exp(-x)     = 1
∫₀¹∫₀¹ exp(-x)   = 1 - exp(-1)


### 5.7  Strategy 7 — Piecewise-monotone substitution

When $g$ has interior critical points, the domain is split there.
Strategy 6 is applied on each monotone piece and the results are summed.

The library also detects **non-differentiable kinks** (zeros of `Abs` arguments
and `sqrt` radicands), so `|x|`, `|sin(x)|`, etc. are handled correctly.


In [19]:
# sin(x) on [0,π]: critical point at π/2
s7a = multiple_integrate(sin(x), (x, 0, pi))
print("∫₀^π sin(x)          =", s7a)   # 2

# cos(x) on [0,2π]: critical point at π
s7b = multiple_integrate(cos(x), (x, 0, 2*pi))
print("∫₀^2π cos(x)         =", s7b)   # 0

# sin²(x) on [0,π]
s7c = multiple_integrate(sin(x)**2, (x, 0, pi))
print("∫₀^π sin²(x)         =", s7c)   # π/2

# Non-analytic f: |x| on [-1,1]  (kink at 0)
s7d = multiple_integrate(Abs(x), (x, -1, 1))
print("∫₋₁¹ |x|             =", s7d)   # 1

# |sin(x)| on [0,2π]
s7e = multiple_integrate(Abs(sin(x)), (x, 0, 2*pi))
print("∫₀^2π |sin(x)|       =", s7e)   # 4

# |cos(x)| on [0,π]
s7f = multiple_integrate(Abs(cos(x)), (x, 0, pi))
print("∫₀^π |cos(x)|        =", s7f)   # 2

# With free dimension: ∫₀^π∫₀¹ sin(x) dy dx = 2
s7g = multiple_integrate(sin(x), (x, 0, pi), (y, 0, 1))
print("∫₀^π∫₀¹ sin(x) dy dx =", s7g)   # 2


∫₀^π sin(x)          = 2
∫₀^2π cos(x)         = 0
∫₀^π sin²(x)         = pi/2
∫₋₁¹ |x|             = 1
∫₀^2π |sin(x)|       = 4
∫₀^π |cos(x)|        = 2
∫₀^π∫₀¹ sin(x) dy dx = 2


### 5.8  Strategy 8 — General non-polynomial, Heaviside layer-cake

For **multi-variable non-polynomial $g$** that is not additively separable (S5).
SymPy evaluates $\int_\Omega\Theta(y-g(\mathbf{x}))\,d\mathbf{x}$ directly.


In [20]:
# Product of single-variable functions (not a sum → not S5)
s8a = multiple_integrate(sin(x)*cos(y), (x, 0, pi/2), (y, 0, pi/2))
print("∫∫ sin(x)cos(y)  [0,π/2]²  =", s8a)   # 1

# Heaviside of a linear combination: area where x+y > 2 in [0,2]²
s8b = multiple_integrate(Heaviside(x + y - 2), (x, 0, 2), (y, 0, 2))
print("∫∫ Θ(x+y-2)  [0,2]²        =", s8b)   # 2


∫∫ sin(x)cos(y)  [0,π/2]²  = 1
∫∫ Θ(x+y-2)  [0,2]²        = 2


### 5.9  Strategy 9 — Fallback (plain iterated integration)

When no earlier strategy applies, `multiple_integrate` falls through to plain
iterated `sympy.integrate`. This handles products like $x\sin(x)$, expressions with
variable limits, and integrands that don't factorise as $f(g(\mathbf{x}))$.


In [21]:
# x·sin(x): not of f(g) form — integration by parts needed
s9a = multiple_integrate(x * sin(x), (x, 0, pi))
print("∫₀^π x·sin(x)            =", s9a)   # π

# x·log(x)
s9b = multiple_integrate(x * log(x), (x, 0, 1))
print("∫₀¹ x·log(x)             =", s9b)   # -1/4

# Variable limits (triangle)
s9c = multiple_integrate(x + y, (y, 0, 1-x), (x, 0, 1))
print("∫∫_triangle (x+y)        =", s9c)   # 1/3

# sin(x)·cos(y) on [0,π]² — separable product that integrates to 0
s9d = multiple_integrate(sin(x)*cos(y), (x, 0, pi), (y, 0, pi))
print("∫∫ sin(x)·cos(y)  [0,π]² =", s9d)   # 0


∫₀^π x·sin(x)            = pi
∫₀¹ x·log(x)             = -1/4
∫∫_triangle (x+y)        = 1/3
∫∫ sin(x)·cos(y)  [0,π]² = 0


### 5.10  Mixed and cross-cutting examples

These combine multiple hard properties: non-analytic $f$, non-polynomial $g$,
2-D non-rectangular domains, and improper integrals.


In [22]:
# Non-analytic f + non-polynomial g: |x - y| on [0,1]²
m1 = multiple_integrate(Abs(x - y), (x, 0, 1), (y, 0, 1))
print("∫∫ |x-y|  [0,1]²              =", m1)   # 1/3

# Discontinuous f: Heaviside(sin(x) - 1/2) on [0,π]
# sin(x) > 1/2 on (π/6, 5π/6) — length 2π/3
m2 = multiple_integrate(Heaviside(sin(x) - Rational(1, 2)), (x, 0, pi))
print("∫₀^π Θ(sin x - 1/2)          =", m2)   # 2π/3

# exp(-|x|) on [-1,1] = 2(1 - e^{-1})
m3 = multiple_integrate(exp(-Abs(x)), (x, -1, 1))
print("∫₋₁¹ exp(-|x|)               =", simplify(m3))

# 3-D mixed: ∫₀^π∫₀¹∫₀^∞ sin(x)·exp(-y)·exp(-z)
m4 = multiple_integrate(sin(x)*exp(-y)*exp(-z), (x,0,pi),(y,0,1),(z,0,oo))
print("∫₀^π∫₀¹∫₀^∞ sin·e⁻ʸ·e⁻ᶻ    =", simplify(m4))   # 2(1 - 1/e)

# Physics: moment of inertia of unit cube about z-axis
m5 = multiple_integrate(x**2 + y**2, (x,0,1),(y,0,1),(z,0,1))
print("I_z of unit cube              =", m5)   # 2/3

# Probability: E[(X+Y)²] for X,Y ~ Uniform[0,1]
m6 = multiple_integrate((x+y)**2, (x,0,1),(y,0,1))
print("E[(X+Y)²]                     =", m6)   # 7/6


∫∫ |x-y|  [0,1]²              = 1/3
∫₀^π Θ(sin x - 1/2)          = 2*pi/3
∫₋₁¹ exp(-|x|)               = 2 - 2*exp(-1)
∫₀^π∫₀¹∫₀^∞ sin·e⁻ʸ·e⁻ᶻ    = 2 - 2*exp(-1)
I_z of unit cube              = 2/3
E[(X+Y)²]                     = 7/6


### 5.11  Convergent and divergent improper integrals

`multiple_integrate` does not pre-screen for convergence — divergent integrals
propagate as `oo` or remain unevaluated.


In [23]:
# Convergent: p-test ∫_1^∞ x^{-2} dx = 1  (p = -2 < -1)
c1 = multiple_integrate(x**(-2), (x, 1, oo))
print("∫₁^∞ x⁻² dx =", c1)   # 1

# Convergent: integrable singularity ∫_0^1 x^{-1/2} dx = 2
c2 = multiple_integrate(x**Rational(-1, 2), (x, 0, 1))
print("∫₀¹ x^(-1/2) dx =", c2)   # 2

# Convergent: log singularity ∫_0^1 log(x) dx = -1
c3 = multiple_integrate(log(x), (x, 0, 1))
print("∫₀¹ log(x) dx =", c3)   # -1

# Divergent: p-test boundary ∫_1^∞ 1/x dx = ∞
d1 = multiple_integrate(1/x, (x, 1, oo))
print("∫₁^∞ 1/x dx =", d1)   # oo

# Divergent: non-integrable singularity ∫_0^1 1/x dx = ∞
d2 = multiple_integrate(1/x, (x, 0, 1))
print("∫₀¹ 1/x dx =", d2)   # oo

# Divergent: wrong-sign Gaussian
d3 = multiple_integrate(exp(x**2), (x, -oo, oo))
print("∫_ℝ exp(x²) dx =", d3)   # oo


∫₁^∞ x⁻² dx = 1
∫₀¹ x^(-1/2) dx = 2
∫₀¹ log(x) dx = -1
∫₁^∞ 1/x dx = oo
∫₀¹ 1/x dx = oo
∫_ℝ exp(x²) dx = oo


### 5.12  Strategy routing — timing comparison

In [24]:
import time

cases = [
    ("S1  Linear [0,∞)²",      exp(-(x+y)),                   (x,0,oo),(y,0,oo)),
    ("S2  Gaussian ℝ²",         exp(-(x**2+y**2)),              (x,-oo,oo),(y,-oo,oo)),
    ("S3  Half-Gaussian [0,∞)", exp(-x**2),                    (x,0,oo)),
    ("S4  Poly unit square",    (x+y)**3,                      (x,0,1),(y,0,1)),
    ("S5  cos(x+y) [0,π]²",    cos(x+y),                      (x,0,pi),(y,0,pi)),
    ("S6  log(x) [1,e]",        log(x),                        (x,1,E)),
    ("S6  1/(1+x²) [0,1]",      1/(1+x**2),                    (x,0,1)),
    ("S7  sin(x) [0,π]",        sin(x),                        (x,0,pi)),
    ("S7  |x| [-1,1]",          Abs(x),                        (x,-1,1)),
    ("S9  x·sin(x) [0,π]",      x*sin(x),                      (x,0,pi)),
]

for label, integrand, *ranges in cases:
    t0 = time.perf_counter()
    result = multiple_integrate(integrand, *ranges)
    ms = (time.perf_counter() - t0) * 1000
    print(f"{label:30s}  →  {str(result):15s}  ({ms:.1f} ms)")


S1  Linear [0,∞)²               →  1                (1.0 ms)
S2  Gaussian ℝ²                 →  pi               (1.0 ms)
S3  Half-Gaussian [0,∞)         →  sqrt(pi)/2       (3.8 ms)
S4  Poly unit square            →  3/2              (0.7 ms)
S5  cos(x+y) [0,π]²             →  -4               (5.5 ms)
S6  log(x) [1,e]                →  1                (0.2 ms)
S6  1/(1+x²) [0,1]              →  pi/4             (2.6 ms)
S7  sin(x) [0,π]                →  2                (1.0 ms)
S7  |x| [-1,1]                  →  1                (0.2 ms)
S9  x·sin(x) [0,π]              →  pi               (4.3 ms)


## 6  Supported Families — One Example Each

This section gives a **compact tour of the currently supported families** that the solver is designed to handle well.  
These are **not** regression tests; the point is to show one representative exact integral for each family and the kind
of closed form the library can return.

The examples below cover:

1. constants and zero integrands  
2. basic one-dimensional exact integrals  
3. endpoint-singular but convergent integrals  
4. rational full-line integrals  
5. box moments  
6. Gaussian moments  
7. radial / polar-coordinate examples  
8. trigonometric and exponential transform-friendly cases  
9. special-function outputs such as logarithms, arctangents, beta, and gamma values


### 6.1  Constants and zero integrands

A symbolic multiple-integration engine should handle constants and zero integrands immediately. On rectangular regions,
these reduce to geometric volume factors.


In [25]:
const_2d = multiple_integrate(1, (x, 0, 1), (y, 0, 2))
zero_2d  = multiple_integrate(0, (x, -1, 3), (y, 2, 5))

print("∫∫ 1  [0,1]×[0,2] =", const_2d)   # 2
print("∫∫ 0  [-1,3]×[2,5] =", zero_2d)   # 0

∫∫ 1  [0,1]×[0,2] = 2
∫∫ 0  [-1,3]×[2,5] = 0


### 6.2  Basic one-dimensional exact integrals

These are the simplest exact cases, but they are still important because they exercise the same dispatch logic used by
higher-dimensional problems.


In [26]:
basic_poly = multiple_integrate(x**3, (x, 0, 2))
basic_trig = multiple_integrate(sin(x), (x, 0, pi))
basic_exp  = multiple_integrate(exp(x), (x, 0, 1))

print("∫₀² x³ dx     =", basic_poly)   # 4
print("∫₀^π sin(x) dx =", basic_trig)  # 2
print("∫₀¹ eˣ dx     =", basic_exp)    # E - 1

∫₀² x³ dx     = 4
∫₀^π sin(x) dx = 2
∫₀¹ eˣ dx     = -1 + E


### 6.3  Endpoint-singular but convergent integrals

Not every singular integrand diverges. These examples are finite even though the integrand is unbounded at an endpoint.


In [27]:
singular_a = multiple_integrate(x**Rational(-1, 2), (x, 0, 1))
singular_b = multiple_integrate(log(x), (x, 0, 1))
singular_c = multiple_integrate(1 / sqrt(1 - x), (x, 0, 1))

print("∫₀¹ x^(-1/2) dx      =", singular_a)  # 2
print("∫₀¹ log(x) dx        =", singular_b)  # -1
print("∫₀¹ 1/sqrt(1-x) dx   =", singular_c)  # 2

∫₀¹ x^(-1/2) dx      = 2
∫₀¹ log(x) dx        = -1
∫₀¹ 1/sqrt(1-x) dx   = 2


### 6.4  Rational full-line integrals

These are classic exact integrals whose answers involve \(\pi\). They are good representatives for rational kernels on
the whole real line.


In [28]:
rat_a = multiple_integrate(1 / (x**2 + 1), (x, -oo, oo))
rat_b = multiple_integrate(1 / (x**4 + 1), (x, -oo, oo))

print("∫_ℝ dx/(x²+1)   =", rat_a)  # π
print("∫_ℝ dx/(x⁴+1)   =", rat_b)  # π/sqrt(2)

∫_ℝ dx/(x²+1)   = pi
∫_ℝ dx/(x⁴+1)   = sqrt(2)*pi/2


### 6.5  Box moments

On product regions such as rectangles and boxes, polynomial moments are among the most natural exact multiple integrals.


In [29]:
box_a = multiple_integrate(x**2 * y**3, (x, 0, 1), (y, 0, 1))
box_b = multiple_integrate(x**2 * y**2, (x, -1, 1), (y, -1, 1))

print("∫∫ x²y³  [0,1]²     =", box_a)  # 1/12
print("∫∫ x²y²  [-1,1]²    =", box_b)  # 4/9

∫∫ x²y³  [0,1]²     = 1/12
∫∫ x²y²  [-1,1]²    = 4/9


### 6.6  Gaussian moments

Gaussian integrals are one of the central exact families in analysis, probability, and physics. The multidimensional
examples below also illustrate separability and even/odd moment structure.


In [30]:
gauss_1d = multiple_integrate(exp(-x**2), (x, -oo, oo))
gauss_1d_moment = multiple_integrate(x**2 * exp(-x**2), (x, -oo, oo))
gauss_2d = multiple_integrate(exp(-(x**2 + y**2)), (x, -oo, oo), (y, -oo, oo))

print("∫_ℝ e^(-x²) dx              =", gauss_1d)         # sqrt(pi)
print("∫_ℝ x² e^(-x²) dx           =", gauss_1d_moment)  # sqrt(pi)/2
print("∫∫_ℝ² e^(-(x²+y²)) dx dy    =", gauss_2d)         # pi

∫_ℝ e^(-x²) dx              = sqrt(pi)
∫_ℝ x² e^(-x²) dx           = sqrt(pi)/2
∫∫_ℝ² e^(-(x²+y²)) dx dy    = pi


### 6.7  Radial / polar-coordinate examples

The current solver does **not** implement a completely general automatic polar-coordinate engine, but radial families are
still very natural examples to include in the notebook. Writing them directly in polar coordinates yields compact exact
results.


In [31]:
disk_area = integrate(1 * r, (r, 0, 1), (theta, 0, 2*pi))
disk_x2 = integrate((r*cos(theta))**2 * r, (r, 0, 1), (theta, 0, 2*pi))
disk_radial = integrate(r**2 * r, (r, 0, 1), (theta, 0, 2*pi))

print("∬_{x²+y²≤1} 1 dA           =", simplify(disk_area))    # pi
print("∬_{x²+y²≤1} x² dA          =", simplify(disk_x2))      # pi/4
print("∬_{x²+y²≤1} (x²+y²) dA     =", simplify(disk_radial))  # pi/2

∬_{x²+y²≤1} 1 dA           = pi
∬_{x²+y²≤1} x² dA          = pi/4
∬_{x²+y²≤1} (x²+y²) dA     = pi/2


### 6.8  Trigonometric and exponential transform-friendly cases

Sums such as \(x+y\) or damped trigonometric/exponential expressions often simplify after convolution-style reasoning,
trigonometric identities, or straightforward separability.


In [32]:
trig_sep = multiple_integrate(sin(x) * sin(y), (x, 0, pi), (y, 0, pi))
trig_sum = multiple_integrate(cos(x + y), (x, 0, pi), (y, 0, pi))
exp_sum = multiple_integrate(exp(-(x + y)), (x, 0, oo), (y, 0, oo))
damped_cos = multiple_integrate(exp(-x) * cos(x), (x, 0, oo))

print("∫∫ sin(x)sin(y)  [0,π]²    =", trig_sep)   # 4
print("∫∫ cos(x+y)  [0,π]²        =", trig_sum)   # -4
print("∫∫ e^(-(x+y))  [0,∞)²      =", exp_sum)    # 1
print("∫₀^∞ e^(-x) cos(x) dx      =", damped_cos) # 1/2

∫∫ sin(x)sin(y)  [0,π]²    = 4
∫∫ cos(x+y)  [0,π]²        = -4
∫∫ e^(-(x+y))  [0,∞)²      = 1
∫₀^∞ e^(-x) cos(x) dx      = 1/2


### 6.9  Special-function outputs

A useful symbolic integrator should return exact answers involving logarithms, inverse trigonometric functions, beta
functions, and gamma functions when those are the natural closed forms.


In [33]:
spec_log = multiple_integrate(1 / (1 + x), (x, 0, 1))
spec_atan = multiple_integrate(1 / (1 + x**2), (x, 0, 1))
beta_ex = multiple_integrate(x**2 * (1 - x)**3, (x, 0, 1))
gamma_ex = multiple_integrate(sqrt(x) * exp(-x), (x, 0, oo))

print("∫₀¹ dx/(1+x)               =", spec_log)  # log(2)
print("∫₀¹ dx/(1+x²)              =", spec_atan) # pi/4
print("∫₀¹ x²(1-x)³ dx            =", beta_ex)   # 1/60
print("∫₀^∞ sqrt(x)e^(-x) dx      =", gamma_ex)  # sqrt(pi)/2

∫₀¹ dx/(1+x)               = log(2)
∫₀¹ dx/(1+x²)              = pi/4
∫₀¹ x²(1-x)³ dx            = 1/60
∫₀^∞ sqrt(x)e^(-x) dx      = sqrt(pi)/2


### 6.10  A note on dependent bounds

The project can successfully evaluate some iterated integrals whose inner bounds depend on outer variables when the
integral is easy in the order given, but the current implementation should **not** be viewed as a full general-purpose
engine for geometric region rewriting or arbitrary order reversal.

For that reason, this notebook keeps the supported-family overview focused on families that are currently well supported
and stable in ordinary use.


## 7  References

### Textbooks

1. **Apostol, T. M.** (1969). *Calculus, Vol. 2: Multi-Variable Calculus and Linear Algebra*. Wiley.  
   Rigorous treatment of multiple integrals and Fubini's theorem.

2. **Rudin, W.** (1987). *Real and Complex Analysis* (3rd ed.). McGraw-Hill.  
   Chapter 8: Integration on product spaces, Fubini–Tonelli theorem.

3. **Evans, L. C., & Gariepy, R. F.** (2015). *Measure Theory and Fine Properties of Functions* (revised ed.). CRC Press.  
   Co-area formula: Theorem 3.11.

4. **Folland, G. B.** (1999). *Real Analysis: Modern Techniques and Their Applications* (2nd ed.). Wiley.  
   Layer-cake representation: Proposition 6.16.

5. **Federer, H.** (1969). *Geometric Measure Theory*. Springer.  
   Original formulation of the co-area formula.

### Papers & technical references

6. **Bronstein, M.** (2005). *Symbolic Integration I: Transcendental Functions* (2nd ed.). Springer.  
   Algorithms underlying SymPy's univariate `integrate`.

7. **Moses, J.** (1971). Symbolic integration: The stormy decade. *Communications of the ACM*, 14(8), 548–560.  
   Historical context for computer algebra integration algorithms.

8. **Risch, R. H.** (1969). The problem of integration in finite terms. *Transactions of the American Mathematical Society*, 139, 167–189.  
   The Risch algorithm used for indefinite integration in SymPy.

### Software

9. **SymPy Development Team** (2023). *SymPy: Python library for symbolic mathematics*. https://www.sympy.org  

### Online resources

10. Evans, L. (2010). *Measure theory and integration* (lecture notes). UC Berkeley.  
    Clear exposition of the layer-cake formula with proofs.

11. Strichartz, R. S. (1994). *A Guide to Distribution Theory and Fourier Transforms*. CRC Press.  
    Heaviside function, delta distributions, and their use in integration.
